# ASR Assignment 2025-26

This notebook has been provided as a template to get you started on the assignment.  Feel free to use it for your development, or do your development directly in Python.

You can find a full description of the assignment [here](https://www.inf.ed.ac.uk/teaching/courses/asr/coursework-2026.html).

You are provided with two Python modules `observation_model.py` and `wer.py`.  The first was described in [Lab 3](https://github.com/geoph9/asr_labs/blob/main/asr_lab3_4.ipynb).  The second can be used to compute the number of substitution, deletion and insertion errors between ASR output and a reference text.

It can be used as follows:

```python
import wer

my_refence = 'A B C'
my_output = 'A C C D'

wer.compute_alignment_errors(my_reference, my_output)
```

This produces a tuple $(s,d,i)$ giving counts of substitution,
deletion and insertion errors respectively - in this example (1, 0, 1).  The function accepts either two strings, as in the example above, or two lists.  Matching is case sensitive.

## Template code

Assuming that you have already made a function to generate an WFST, `create_wfst()` and a decoder class, `MyViterbiDecoder`, you can perform recognition on all the audio files as follows:


In [21]:
import glob
import os
import wer
import observation_model
import openfst_python as fst
from decoder import MyViterbiDecoder
from utils import parse_lexicon, generate_symbol_tables
from wfst_build import build_hlg
import math
import time
from collections import Counter, defaultdict

def compute_unigram_probs(wav_files):
    counts = Counter()
    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        counts.update(words)
    total = sum(counts.values())
    return {word: count / total for word, count in counts.items()}

def compute_bigram_probs(wav_files, vocab, smoothing=1):
    bigram_counts = defaultdict(Counter)
    unigram_counts = Counter()

    for wav_file in wav_files:
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f:
            words = f.readline().strip().split()
        for i in range(len(words) - 1):
            bigram_counts[words[i]][words[i+1]] += 1
            unigram_counts[words[i]] += 1
        if words:
            unigram_counts[words[-1]] += 1

    V = len(vocab)
    bigram_probs = {}
    for w1 in vocab:
        bigram_probs[w1] = {}
        for w2 in vocab:
            count = bigram_counts[w1][w2] + smoothing
            total = unigram_counts[w1] + smoothing * V
            bigram_probs[w1][w2] = count / total

    return bigram_probs

def compute_interpolated_probs(unigram_probs, bigram_probs, vocab, lam=0.5):  # ADDED
    """
    Interpolated LM: P(w2|w1) = lam * P_bigram(w2|w1) + (1-lam) * P_unigram(w2)
    lam=1.0 -> pure bigram, lam=0.0 -> pure unigram
    """
    interpolated = {}
    for w1 in vocab:
        interpolated[w1] = {}
        for w2 in vocab:
            p_bigram = bigram_probs[w1][w2]
            p_unigram = unigram_probs.get(w2, 1.0 / len(vocab))
            interpolated[w1][w2] = lam * p_bigram + (1 - lam) * p_unigram
    return interpolated

def create_wfst_interpolated(lexicon, word_table, phone_table, state_table, interpolated_probs, unigram_probs, n_states=3, stay_cost=0.9, final_prob=0.1):  # ADDED
    f = fst.Fst('log')
    f.set_input_symbols(state_table)
    f.set_output_symbols(word_table)

    trans_cost = 1.0 - stay_cost

    # one start state + one loop state per word
    start_state = f.add_state()
    f.set_start(start_state)

    word_loop_states = {}
    for word in lexicon:
        word_loop_states[word] = f.add_state()
        f.set_final(word_loop_states[word])

    # silence HMM
    n_sil_states = 5
    sil_states = [f.add_state() for _ in range(n_sil_states)]
    for i, sil_state in enumerate(sil_states):
        sil_label = state_table.find(f"sil_{i+1}")
        f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(stay_cost)), sil_state))
        if i < n_sil_states - 1:
            f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(trans_cost)), sil_states[i+1]))
        else:
            f.add_arc(sil_state, fst.Arc(sil_label, 0, fst.Weight('log', -math.log(trans_cost)), start_state))

    # from start_state enter any word weighted by unigram (no previous word context)
    for word in lexicon:
        p_unigram = unigram_probs.get(word, 1.0 / len(lexicon))
        f.add_arc(start_state, fst.Arc(0, 0, fst.Weight('log', -math.log(p_unigram)), word_loop_states[word]))

    # optional silence from start
    f.add_arc(start_state, fst.Arc(0, 0, fst.Weight.One(f.weight_type()), sil_states[0]))

    for word, phones in lexicon.items():
        word_id = word_table.find(word)

        # from each word's loop state, transition to next word using interpolated prob
        for next_word in lexicon:
            p_interp = interpolated_probs[word][next_word]
            f.add_arc(word_loop_states[word], fst.Arc(0, 0, fst.Weight('log', -math.log(p_interp)), word_loop_states[next_word]))

        current_state = word_loop_states[word]

        for phone_idx, phone in enumerate(phones):
            for i in range(1, n_states + 1):
                state_sym = f"{phone}_{i}"
                in_label = state_table.find(state_sym)

                sl_weight = fst.Weight('log', -math.log(stay_cost))
                f.add_arc(current_state, fst.Arc(in_label, 0, sl_weight, current_state))

                next_state = f.add_state()
                fw_weight = fst.Weight('log', -math.log(trans_cost))

                if phone_idx == len(phones) - 1 and i == n_states:
                    out_label = word_id
                else:
                    out_label = 0

                f.add_arc(current_state, fst.Arc(in_label, out_label, fw_weight, next_state))
                current_state = next_state

        final_weight = fst.Weight('log', -math.log(final_prob))
        f.add_arc(current_state, fst.Arc(0, 0, final_weight, word_loop_states[word]))   # bypass silence
        f.add_arc(current_state, fst.Arc(0, 0, final_weight, sil_states[0]))             # enter silence

    return f

def run_decode(f, wav_files, beam=100, max_states=None):
    total_substitutions = 0
    total_deletions = 0
    total_insertions = 0
    total_words = 0
    total_forward_computations = 0
    total_decode_time = 0.0
    total_backtrace_time = 0.0

    for wav_file in wav_files:
        decoder = MyViterbiDecoder(f, wav_file, beam=beam, max_states=max_states)

        t0 = time.perf_counter()
        decoder.decode()
        decode_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        (state_path, words) = decoder.backtrace()
        backtrace_time = time.perf_counter() - t0

        total_decode_time += decode_time
        total_backtrace_time += backtrace_time
        total_forward_computations += decoder.forward_computation_count

        words_str = ' '.join(words)
        transcription_file = os.path.splitext(wav_file)[0] + '.txt'
        with open(transcription_file, 'r') as f_txt:
            transcription = f_txt.readline().strip()
        error_counts = wer.compute_alignment_errors(transcription, words_str)
        word_count = len(transcription.split())

        total_substitutions += error_counts[0]
        total_deletions += error_counts[1]
        total_insertions += error_counts[2]
        total_words += word_count

    overall_wer = (total_substitutions + total_deletions + total_insertions) / total_words
    return {
        'wer': overall_wer,
        'sub': total_substitutions,
        'del': total_deletions,
        'ins': total_insertions,
        'fwd': total_forward_computations,
        'decode_time': total_decode_time,
        'backtrace_time': total_backtrace_time
    }

def print_results(label, r, num_states, num_arcs):
    print(f"=== {label} ===")
    print(f"  WER                        : {r['wer']:.2%}")
    print(f"  Total substitutions        : {r['sub']}")
    print(f"  Total deletions            : {r['del']}")
    print(f"  Total insertions           : {r['ins']}")
    print(f"  Total forward computations : {r['fwd']}")
    print(f"  Total decode time          : {r['decode_time']:.4f}s")
    print(f"  Total backtrace time       : {r['backtrace_time']:.4f}s")
    print(f"  Total time                 : {r['decode_time'] + r['backtrace_time']:.4f}s")
    print(f"  WFST states                : {num_states}")
    print(f"  WFST arcs                  : {num_arcs}\n")


lex = parse_lexicon("lexicon.txt")
word_table, phone_table, state_table = generate_symbol_tables(lex)

for i in range(1, 6):
    state_table.add_symbol(f"sil_{i}")

stay_cost = 0.9
final_prob = 0.1
beam = 50
beam_hlg = 50
max_states_hlg = 200
lg_optimize = True

wav_files = glob.glob('/group/teaching/asr/labs/recordings/*.wav')
vocab = list(lex.keys())

unigram_probs = compute_unigram_probs(wav_files)
bigram_probs = compute_bigram_probs(wav_files, vocab)

# --- Sweep over lambda: monolithic vs H ∘ (L ∘ G) tree lexicon + LM look-ahead ---
best_wer_mono = float('inf')
best_lam_mono = None

for lam in [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]:
    interpolated_probs = compute_interpolated_probs(unigram_probs, bigram_probs, vocab, lam=lam)

    baseline_wfst = create_wfst_interpolated(lex, word_table, phone_table, state_table,
                                             interpolated_probs, unigram_probs,
                                             stay_cost=stay_cost, final_prob=final_prob)

    num_states = baseline_wfst.num_states()
    num_arcs = sum(1 for s in baseline_wfst.states() for _ in baseline_wfst.arcs(s))
    r = run_decode(baseline_wfst, wav_files, beam=beam)
    print_results(f"Monolithic interpolated WFST (lambda={lam:.1f})", r, num_states, num_arcs)

    if r['wer'] < best_wer_mono:
        best_wer_mono = r['wer']
        best_lam_mono = lam

    hlg_wfst = build_hlg(
        lex, vocab, word_table, phone_table, state_table,
        interpolated_probs, unigram_probs,
        stay_cost=stay_cost,
        lg_optimize=lg_optimize,
        add_disambig=True,
        verbose_graph_ops=False,
    )
    nh = hlg_wfst.num_states()
    ah = sum(1 for s in hlg_wfst.states() for _ in hlg_wfst.arcs(s))
    rh = run_decode(hlg_wfst, wav_files, beam=beam_hlg, max_states=max_states_hlg)
    print_results(f"Tree lexicon + LM look-ahead HLG (lambda={lam:.1f})", rh, nh, ah)

print(f"=== Best monolithic: lambda={best_lam_mono:.1f}, WER={best_wer_mono:.2%} ===")

=== Monolithic interpolated WFST (lambda=0.0) ===
  WER                        : 48.05%
  Total substitutions        : 389
  Total deletions            : 279
  Total insertions           : 132
  Total forward computations : 10683986
  Total decode time          : 358.5529s
  Total backtrace time       : 0.0659s
  Total time                 : 358.6188s
  WFST states                : 121
  WFST arcs                  : 351



IndexError: list assignment index out of range